# latex-slm: train a small LaTeX model on Colab
Runtime > Change runtime type > GPU (a T4 is enough for a 0.5B LoRA; A100/L4 is faster).
Run the cells top to bottom. Data and checkpoints live on Google Drive so a disconnect does not lose work.

Colab secrets (key icon in the left bar): `GITHUB_TOKEN` (required, the repo is private), optional `ANTHROPIC_API_KEY` (better task descriptions) and `HF_TOKEN`.

In [ ]:
!nvidia-smi -L
import os
from google.colab import userdata, drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/latex-slm'
os.makedirs(WORK + '/data', exist_ok=True)
os.makedirs(WORK + '/outputs', exist_ok=True)

In [ ]:
REPO = 'techparamveer/latex-slm'
try:
    tok = userdata.get('GITHUB_TOKEN')
except Exception:
    tok = None
url = f'https://{tok}@github.com/{REPO}.git' if tok else f'https://github.com/{REPO}.git'
if not os.path.exists('/content/latex-slm'):
    !git clone -q {url} /content/latex-slm
%cd /content/latex-slm
!git pull -q
!ln -sfn {WORK}/data data
!ln -sfn {WORK}/outputs outputs

In [ ]:
# TeX Live + pandoc (about 4-6 minutes). Needed to build the data, not to train.
!apt-get -qq update
!apt-get -qq install -y texlive-latex-base texlive-latex-recommended texlive-latex-extra texlive-fonts-recommended texlive-science pandoc > /dev/null
!latex --version | head -1
!pandoc --version | head -1

In [ ]:
!pip -q install -r requirements.txt
for k in ['ANTHROPIC_API_KEY', 'HF_TOKEN']:
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        print('no secret', k)

## 1. Build snippets
Stream arXiv LaTeX, extract, dedupe, compile-filter. Starter scale is 1,500 papers; raise `--max_papers` once it works. Prints the real compile pass rate.

In [ ]:
!python scripts/02_build_snippets.py --max_papers 1500 --workers 2

## 2. Build the 4 task datasets (fix, gen, fim, conv)
Uses Claude for descriptions if `ANTHROPIC_API_KEY` is set, otherwise a local Qwen 1.5B. Includes an automated paper-leakage check.

In [ ]:
!python scripts/03_build_tasks.py --n1 4000 --n2 3000 --n3_max 6000 --n4 2500

## 3. Smoke test (30 steps), then the full run

In [ ]:
!python scripts/04_train.py --out outputs/smoke --max_steps 30 --limit 500

In [ ]:
!python scripts/04_train.py --out outputs/run1 --resume

## 4. Evaluate base vs tuned on held-out TEST papers

In [ ]:
!python scripts/05_eval.py --model Qwen/Qwen2.5-Coder-0.5B-Instruct --tag base
!python scripts/05_eval.py --model outputs/run1/merged --tag tuned
!cat outputs/eval/results.md

## 5. Try it

In [ ]:
!python scripts/06_infer.py --model outputs/run1/merged --task gen --text "a 3x3 matrix with ones on the diagonal and zeros elsewhere, in brackets"